# Notebook 04 — Compressed Sensing Image Reconstruction (Part B)

**Dataset:** Fashion-MNIST (28×28 grayscale, d=784)

**Sparsity domain:** 2D-DCT coefficients α = Ψ x

**Measurement model:** y = A α + n,  A ∈ ℝ^{m×d}

**Measurement ratios:** m/d ∈ {0.125, 0.25, 0.5}

Each model is evaluated on:
- NMSE (dB) in DCT domain
- PSNR and SSIM in pixel domain (after IDCT)
- Visual comparison grid

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
import pandas as pd

from src.data.image_loader     import build_image_cs_dataloaders
from src.operators.dct_operators import idct2_flat
from src.models.ista           import ISTA
from src.models.fista          import FISTA
from src.models.lista          import LISTA
from src.models.alista         import ALISTA
from src.models.hyperlista     import HyperLISTA
from src.training.trainer      import train
from src.training.tuner        import tune_hyperlista
from src.evaluation.metrics    import evaluate_model
from src.evaluation.visualizer import plot_nmse_vs_layers, get_layerwise_nmse, plot_image_comparison

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_LAYERS = 16
D        = 784   # 28 × 28
print('Device:', DEVICE)

## 1. Helper: build models for a given sensing matrix A

In [ ]:
def build_and_train_models(A, train_loader, val_loader, n_epochs=30):
    """
    Instantiate all five models for a given A and train the learnable ones.
    Returns dict {name: model}.
    """
    m = A.shape[0]
    print(f'\nBuilding models for m={m}, d={A.shape[1]}  (ratio {m/A.shape[1]:.3f})')

    ista  = ISTA(A, lam=0.05, n_iter=N_LAYERS).to(DEVICE)
    fista = FISTA(A, lam=0.05, n_iter=N_LAYERS).to(DEVICE)

    lista = LISTA(A, n_layers=N_LAYERS).to(DEVICE)
    print('Training LISTA...')
    train(lista, train_loader, val_loader,
          n_epochs=n_epochs, lr=1e-3, device=DEVICE, patience=10, verbose=False)

    alista = ALISTA(A, n_layers=N_LAYERS, W_iters=1000).to(DEVICE)
    print('Training ALISTA...')
    train(alista, train_loader, val_loader,
          n_epochs=n_epochs, lr=5e-4, device=DEVICE, patience=10, verbose=False)

    hyperlista = HyperLISTA(A, n_layers=N_LAYERS, W_iters=1000).to(DEVICE)
    print('Tuning HyperLISTA...')
    tune_hyperlista(hyperlista, val_loader, DEVICE,
                    coarse_points=5, fine_points=5, verbose=False)

    return {
        'ISTA': ista, 'FISTA': fista,
        'LISTA': lista, 'ALISTA': alista,
        'HyperLISTA': hyperlista,
    }

## 2. Run all three measurement ratios

In [ ]:
RATIOS    = [0.125, 0.25, 0.5]
all_models = {}
all_loaders = {}

for ratio in RATIOS:
    A_r, Psi, tr_loader, te_loader = build_image_cs_dataloaders(
        measurement_ratio=ratio,
        sigma=0.0,
        batch_size=128,
        device=DEVICE,
        data_root='./data',
    )
    # Use a subset of training data for faster experiments
    # (reduce n_epochs or batch size for quick sanity-check)
    models_r = build_and_train_models(A_r, tr_loader, te_loader, n_epochs=20)
    all_models[ratio]  = models_r
    all_loaders[ratio] = (A_r, tr_loader, te_loader)

print('\nAll ratios done.')

## 3. NMSE vs. Layer (m/d = 0.25)

In [ ]:
RATIO = 0.25
A_r, _, te_loader = all_loaders[RATIO]
models_r = all_models[RATIO]

# DataLoader for get_layerwise_nmse: must yield (b, x_true)
# Here b=y (measurements), x_true=alpha (DCT coefficients)
from torch.utils.data import DataLoader

class AlphaLoader:
    """Wrapper that makes (y, alpha, x) dataset look like (y, alpha)."""
    def __init__(self, loader): self.loader = loader
    def __iter__(self):
        for y, alpha, x in self.loader:
            yield y, alpha
    def __len__(self): return len(self.loader)

alpha_loader = AlphaLoader(te_loader)

nmse_curves = {}
for name, model in models_r.items():
    nmse_curves[name] = get_layerwise_nmse(model, alpha_loader, DEVICE)

fig = plot_nmse_vs_layers(
    nmse_curves,
    title=f'Fashion-MNIST CS Reconstruction — m/d = {RATIO}',
)
fig.savefig(f'nmse_vs_layers_cs_{RATIO}.pdf', bbox_inches='tight')
plt.show()

## 4. Full Metrics Table (all ratios)

In [ ]:
rows = []
for ratio in RATIOS:
    _, _, te_loader = all_loaders[ratio]
    for name, model in all_models[ratio].items():
        res = evaluate_model(
            model, te_loader, DEVICE,
            image_mode=True, H=28, W=28, max_val=1.0,
        )
        rows.append({'ratio': ratio, 'Model': name, **res})

df = pd.DataFrame(rows).set_index(['ratio', 'Model'])
print(df[['nmse_db', 'psnr', 'ssim', 'runtime_ms', 'n_params']].round(3).to_string())

## 5. Visual Comparison Grid (m/d = 0.25)

In [ ]:
RATIO = 0.25
_, _, te_loader = all_loaders[RATIO]
models_r = all_models[RATIO]

fig = plot_image_comparison(
    models_r, te_loader, DEVICE,
    H=28, W=28, n_samples=4,
    save_path=f'image_comparison_{RATIO}.pdf',
)
plt.show()

## 6. PSNR vs. Measurement Ratio Bar Chart

In [ ]:
model_names = list(next(iter(all_models.values())).keys())
x = range(len(RATIOS))
width = 0.15
colors = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(9, 5))
for i, name in enumerate(model_names):
    psnr_vals = [
        evaluate_model(all_models[r][name], all_loaders[r][2], DEVICE,
                       image_mode=True, measure_time=False)['psnr']
        for r in RATIOS
    ]
    offset = (i - len(model_names)/2 + 0.5) * width
    ax.bar([v + offset for v in x], psnr_vals, width=width, label=name, color=colors[i])

ax.set_xticks(list(x))
ax.set_xticklabels([f'm/d={r}' for r in RATIOS])
ax.set_ylabel('PSNR (dB)', fontsize=12)
ax.set_title('PSNR vs. Measurement Ratio — Fashion-MNIST CS', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.4)
fig.tight_layout()
fig.savefig('psnr_vs_ratio.pdf', bbox_inches='tight')
plt.show()